<a href="https://colab.research.google.com/github/sayandxzzz/sayan/blob/main/simplechatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import nltk
import random
import json
import string
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
nltk.download("punkt")

In [ ]:
intents = {
    "greetings": {
        "patterns": ["hi", "hello", "hey", "good morning", "good evening"],
        "responses": ["Hello! How can I help you?", "Hey! What can I do for you?"]
    },
    "name": {
        "patterns": ["what is your name", "who are you", "tell me your name"],
        "responses": ["I am your AI chatbot assistant.", "You can call me SmartBot."]
    },
    "help": {
        "patterns": ["help me", "what can you do", "how can you help me"],
        "responses": ["I can chat with you, answer basic questions, and remember your name."]
    },
    "thanks": {
        "patterns": ["thank you", "thanks", "thank you so much"],
        "responses": ["You're welcome!", "Glad to help!"]
    },
    "bye": {
        "patterns": ["bye", "goodbye", "see you"],
        "responses": ["Goodbye!", "See you soon!"]
    }
}

In [ ]:
patterns = []
tags = []
for tag, data in intents.items():
    for pattern in data["patterns"]:
        patterns.append(pattern)
        tags.append(tag)

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(patterns)

In [ ]:
user_memory = {
    "name": None
}
def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text
def get_response(user_input):
    user_input = clean_text(user_input)

    if "my name is" in user_input:
        name = user_input.replace("my name is", "").strip()
        user_memory["name"] = name
        return f"Nice to meet you, {name}!"

    if "what is my name" in user_input:
        if user_memory["name"]:
            return f"Your name is {user_memory['name']}."
        else:
            return "I don't know your name yet."

    user_vec = vectorizer.transform([user_input])
    similarity = cosine_similarity(user_vec, X)

    best_match_index = np.argmax(similarity)
    best_score = similarity[0][best_match_index]

    if best_score < 0.25:
        return "Sorry, I don't understand. Please ask in another way."

    tag = tags[best_match_index]
    return random.choice(intents[tag]["responses"])

In [ ]:
print("SmartBot: Hello! Type 'bye' to exit.")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["bye", "exit", "quit"]:
        print("SmartBot: Goodbye!")
        break

    response = get_response(user_input)
    print("SmartBot:", response)